<a href="https://colab.research.google.com/github/Alisherbakh/jaydariGPT/blob/main/jaydari_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install transformers torch bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 44.3 MB/s  0:00:01


In [10]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
import torch

In [12]:
model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'      # Hugging Face'dagi model nomi: "muallif/model_nomi"
tokenizer = AutoTokenizer.from_pretrained(model_id)  # Shu modelga mos tayyor tokenizerni (lug'at + bo'lish qoidalari) yuklab olamiz

#print('Vocab size:', tokenizer.vocab_size)               # Lug'atdagi tokenlar soni → 32000
#print('Special tokens:', tokenizer.special_tokens_map)   # Maxsus tokenlar → {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '</s>'}


# quantization
bnb_config = BitsAndBytesConfig(                  # Modelni siqish (quantization) sozlamalari
    load_in_4bit=True,                            # Og'irliklarni 4 bitga siqib yuklaymiz (odatda 16 bit bo'ladi)
    bnb_4bit_use_double_quant=True,               # Siqish uchun kerak bo'lgan yordamchi sonlarni ham siqamiz → yana ozgina xotira tejaladi
    bnb_4bit_compute_dtype=torch.bfloat16,        # Hisoblash paytida sonlar vaqtincha bfloat16 ga ochiladi (aniqlik uchun)
    bnb_4bit_quant_type='nf4'                     # Siqish usuli: NF4 — neyron tarmoq og'irliklari uchun maxsus 4 bitli format
)

bnb_config                                        # Sozlamalarni ekranga chiqaramiz (tekshirish uchun)

model = AutoModelForCausalLM.from_pretrained(     # Matn yozuvchi (keyingi tokenni bashorat qiluvchi) tayyor modelni yuklaymiz
    model_id,                                     # Qaysi model (tokenizer bilan bir xil bo'lishi shart)
    quantization_config=bnb_config,               # Yuqoridagi 4 bitli siqish sozlamalarini qo'llaymiz
    device_map='auto',   # GPU, CPU                 → model qismlarini GPU va CPU'ga avtomatik taqsimlaydi
    dtype=torch.bfloat16                          # Siqilmagan qismlar uchun ma'lumot turi
)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [14]:
# Before Fine-tuning                                   → modelni o'zimizning ma'lumotda o'qitishdan OLDIN qanday javob berishini ko'ramiz
prompt = "Explain what a tokenizer is?"                # Modelga beriladigan savol (prompt)
# prompt = "A tokenizer is a tool in natural language processing that"   → muqobil: gapni boshlab berib, davomini yozdirish

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)   # GPU, CPU   → promptni tensorga aylantirib, model turgan qurilmaga ko'chiramiz

with torch.no_grad():                                  # Gradient hisoblamaymiz (faqat javob olyapmiz, o'qitmayapmiz)
    output_ids = model.generate(                       # Model javob matnini token-token yozadi
        **inputs,                                      # input_ids va attention_mask ni argument qilib beramiz
        max_new_tokens=80,                             # Ko'pi bilan 80 ta YANGI token yozsin
        do_sample=True,                                # Har safar eng ehtimollisini emas, ehtimollik bo'yicha tasodifiy tanlaydi
        temperature=0.7                                # Tasodifiylik darajasi: past → aniqroq, baland → ijodkorroq
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))   # Javob ID'larini matnga aylantirib chiqaramiz (<s>, </s> siz)

# print(model)                                  → butun modelni chiqarish o'chirib qo'yildi (juda uzun)
first_block = model.model.layers[0]             # 22 ta Transformer blokdan birinchisini (0-indeks) olamiz
print('first_block:', first_block)              # Shu bitta blokning ichki tuzilmasini chiqaramiz
print('=======')                                # Ajratuvchi chiziq
print(first_block.self_attn)                    # Blok ichidagi Attention (diqqat) qismini alohida chiqaramiz
print('=======')                                # Ajratuvchi chiziq
print(model.config)                             # Modelning barcha sozlamalarini (o'lchamlar, qatlamlar soni) chiqaramiz

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain what a tokenizer is?
first_block: LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
)
LlamaAttention(
  (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
  (v_proj): Linear4bit(in_features=2048, out_feat

In [15]:
def count_parameters(model):                                  # Modeldagi parametrlar sonini sanaydigan funksiya
    return sum(p.numel() for p in model.parameters())         # Har bir parametr tensoridagi sonlar miqdorini olib, hammasini qo'shamiz

total_params = count_parameters(model)                        # Funksiyani modelimizga qo'llaymiz
print(f"Total parameters (including frozen 4-bit): {total_params:,}")   # Natijani mingliklarni vergul bilan ajratib chiqaramiz → 615,606,272

Total parameters (including frozen 4-bit): 615,606,272
